# 07 — 10-Split Energy Prediction Robustness Evaluation

Trains LSTM and BiLSTM on all 10 random splits to produce stable mean ± std MAE estimates.
CNN anomaly scores are available for all 10 splits.

**Why this notebook exists:**
Notebook 06 used a single split (split_05_seed_55). Neural network results on a single split
are sensitive to weight initialisation and data ordering. This notebook evaluates all 10 splits
to confirm the thesis findings are robust and not artefacts of a lucky split.


## 1. Imports, Paths, and Seeds

In [4]:
import os, gc, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import psutil
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────
BASE_DIR      = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")
FOLDER_VER    = "v3_voltage_random10_issue_flight-removed"
FILE_VER      = "v4_voltage_random10_cleaned"
DATA_DIR      = BASE_DIR / f"model_ready_no_dr_{FOLDER_VER}"
CNN_SCORE_DIR = BASE_DIR / "model_outputs" / f"cnn_lstm_ae_anomaly_score_{FOLDER_VER}"
OUTPUT_DIR    = BASE_DIR / "model_outputs" / "07_10split_energy"
PLOT_DIR      = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ── Target and feature sets ────────────────────────────────────
TARGET = "target_energy_next_60s_wh"

BASE_FEATURES = [
    "battery_status__cell_count",
    "battery_status__current_a",
    "battery_status__current_filtered_a",
    "battery_status__discharged_mah",
    "battery_status__remaining",
    "battery_status__scale",
    "battery_status__voltage_filtered_v",
    "battery_status__voltage_v",
    "cumulative_energy_wh",
    "power_w",
    "time_s",
    "voltage_per_cell",
    "voltage_filtered_per_cell",
]

FEATURE_SETS = {
    "Base_13":           BASE_FEATURES,
    "Base_13+CNN_Score": BASE_FEATURES + ["anomaly_score_cnn"],
}

# ── All 10 splits ──────────────────────────────────────────────
SPLITS = [
    ("split_01_seed_11",  11),
    ("split_02_seed_22",  22),
    ("split_03_seed_33",  33),
    ("split_04_seed_44",  44),
    ("split_05_seed_55",  55),
    ("split_06_seed_66",  66),
    ("split_07_seed_77",  77),
    ("split_08_seed_88",  88),
    ("split_09_seed_99",  99),
    ("split_10_seed_111", 111),
]

# ── Device ─────────────────────────────────────────────────────
device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Device : {device}")
print(f"RAM    : {psutil.virtual_memory().available/1e9:.1f} GB free")
print(f"Splits : {len(SPLITS)}")
print(f"Output : {OUTPUT_DIR}")


Device : mps
RAM    : 3.1 GB free
Splits : 10
Output : /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/07_10split_energy


## 2. Model Definitions

In [5]:
# ── Hyperparameters ──────────────────────────────────────────
SEQ_LEN    = 30
BATCH_SIZE = 512
EPOCHS     = 50     # higher ceiling — early stopping will cut it
PATIENCE   = 10     # more patience than notebook 06 (was 5)


class FastFlightDataset(torch.utils.data.Dataset):
    """Pre-builds all sliding-window sequences at init time."""
    def __init__(self, df, feat_cols, target, seq_len):
        seqs, tgts = [], []
        for _, grp in df.sort_values(["log_id","time_s"]).groupby("log_id", sort=False):
            X = grp[feat_cols].to_numpy(dtype=np.float32)
            y = grp[target].to_numpy(dtype=np.float32)
            n = len(X)
            if n <= seq_len:
                continue
            for i in range(seq_len, n):
                seqs.append(X[i-seq_len:i])
                tgts.append(y[i])
        self.X = torch.from_numpy(np.array(seqs, dtype=np.float32))
        self.y = torch.from_numpy(np.array(tgts,  dtype=np.float32))

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class LSTMRegressor(nn.Module):
    """Single-layer unidirectional LSTM, hidden=64."""
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class BiLSTMRegressor(nn.Module):
    """Single-layer bidirectional LSTM, hidden=64."""
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(hidden*2, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def train_seq_model(model, train_ds, val_ds, ckpt_path,
                    seed, epochs=EPOCHS, patience=PATIENCE):
    """
    Train with Huber loss, Adam, ReduceLROnPlateau, early stopping.
    Saves best checkpoint to ckpt_path. Seed passed in for DataLoader.
    """
    criterion = nn.HuberLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5)

    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, generator=g)
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)

    best_val, pat_ctr = np.inf, 0
    for epoch in range(1, epochs+1):
        model.train()
        tl = [criterion(model(xb.to(device)), yb.to(device)).item()
              for xb, yb in train_loader
              if not (optimizer.zero_grad() or
                      criterion(model(xb.to(device)), yb.to(device)).backward() or
                      torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) or
                      optimizer.step())]

        # cleaner loop
        model.train()
        tl = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb.to(device)), yb.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tl.append(loss.item())
        t_loss = np.mean(tl)

        model.eval()
        vl = []
        with torch.no_grad():
            for xb, yb in val_loader:
                vl.append(criterion(model(xb.to(device)), yb.to(device)).item())
        v_loss = np.mean(vl)
        scheduler.step(v_loss)

        if v_loss < best_val - 1e-5:
            best_val, pat_ctr = v_loss, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            pat_ctr += 1
            if pat_ctr >= patience:
                print(f"    Early stop ep {epoch}  best_val={best_val:.5f}")
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()
    return model, best_val


def predict_seq(model, df, feat_cols, target, seq_len):
    """Batched inference, returns DataFrame with y_true, y_pred, dataset_label."""
    model.eval()
    records = []
    for lid, grp in df.sort_values(["log_id","time_s"]).groupby("log_id"):
        X  = grp[feat_cols].to_numpy(dtype=np.float32)
        y  = grp[target].to_numpy(dtype=np.float32)
        t  = grp["time_s"].to_numpy()
        lbl = grp["dataset_label"].iloc[0]
        n = len(X)
        if n <= seq_len:
            continue
        seqs = np.stack([X[i-seq_len:i] for i in range(seq_len, n)])
        preds = []
        with torch.no_grad():
            for batch in torch.split(torch.tensor(seqs), BATCH_SIZE):
                preds.extend(model(batch.to(device)).cpu().numpy().tolist())
        for p, yt, ts in zip(preds, y[seq_len:], t[seq_len:]):
            records.append({"log_id": lid, "dataset_label": lbl,
                            "time_s": ts, "y_true": float(yt), "y_pred": float(p)})
    return pd.DataFrame(records)


print("Model definitions ready ✓")
print(f"EPOCHS={EPOCHS}  PATIENCE={PATIENCE}  SEQ_LEN={SEQ_LEN}")


Model definitions ready ✓
EPOCHS=50  PATIENCE=10  SEQ_LEN=30


## 3. 10-Split Training Loop

Trains LSTM and BiLSTM (Base_13 and Base_13+CNN_Score) on all 10 splits.
Each split uses its own seed for full reproducibility.
Checkpoints saved per split so this cell can be interrupted and resumed.


LSTM anomaly detection time series Malhotra 2015

In [7]:
all_split_results = []

for split_name, seed in SPLITS:
    print(f"\n{'='*60}")
    print(f"SPLIT: {split_name}  SEED: {seed}")
    print(f"{'='*60}")

    # ── Per-split seeds ───────────────────────────────────────
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

    # ── Load data ─────────────────────────────────────────────
    split_dir = DATA_DIR / split_name
    cnn_dir   = CNN_SCORE_DIR / split_name

    train_df = pd.read_parquet(
        split_dir / f"train_no_dr_scaled_{FILE_VER}_{split_name}.parquet")
    val_df = pd.read_parquet(
        split_dir / f"val_no_dr_scaled_{FILE_VER}_{split_name}.parquet")
    test_df = pd.read_parquet(
        split_dir / f"test_no_dr_scaled_{FILE_VER}_{split_name}.parquet")

    # ── Merge CNN scores ──────────────────────────────────────
    cnn_file = cnn_dir / f"test_cnn_lstm_ae_scored_{split_name}.parquet"
    cnn_test = pd.read_parquet(cnn_file,
                columns=["log_id","time_s","anomaly_score_cnn"])
    test_df  = test_df.sort_values(["log_id","time_s"]).reset_index(drop=True)
    cnn_test = cnn_test.sort_values(["log_id","time_s"]).reset_index(drop=True)

    if len(test_df) == len(cnn_test):
        test_df["anomaly_score_cnn"] = cnn_test["anomaly_score_cnn"].values.astype(np.float32)
        print(f"  CNN scores merged ✓  ({len(test_df):,} rows)")
    else:
        print(f"  WARNING: row mismatch {len(test_df)} vs {len(cnn_test)} — using zeros")
        test_df["anomaly_score_cnn"] = np.float32(0.0)

    for df in [train_df, val_df]:
        df["anomaly_score_cnn"] = np.float32(0.0)

    for df in [train_df, val_df, test_df]:
        for col in df.select_dtypes("float64").columns:
            df[col] = df[col].astype(np.float32)

    print(f"  Train:{train_df.shape} Val:{val_df.shape} Test:{test_df.shape}")

    # ── Train LSTM and BiLSTM on both feature sets ────────────
    for feat_name, feat_cols in FEATURE_SETS.items():
        available = [f for f in feat_cols if f in train_df.columns]

        train_ds = FastFlightDataset(train_df, available, TARGET, SEQ_LEN)
        val_ds   = FastFlightDataset(val_df,   available, TARGET, SEQ_LEN)

        for arch_name, ModelClass in [("LSTM",   LSTMRegressor),
                                       ("BiLSTM", BiLSTMRegressor)]:
            run = f"{arch_name}_{feat_name}_{split_name}"

            # Re-seed before each model init
            torch.manual_seed(seed)
            np.random.seed(seed)
            random.seed(seed)

            model = ModelClass(input_size=len(available)).to(device)

            ckpt_dir = OUTPUT_DIR / f"checkpoints_{split_name}"
            ckpt_dir.mkdir(parents=True, exist_ok=True)
            ckpt_path = ckpt_dir / f"ckpt_{arch_name}_{feat_name}.pt"

            # Skip if checkpoint already exists (resume support)
            if ckpt_path.exists():
                print(f"  SKIP (checkpoint exists): {run}")
                model.load_state_dict(
                    torch.load(ckpt_path, map_location=device, weights_only=True))
                model.eval()
            else:
                print(f"  Training: {run}")
                model, best_val = train_seq_model(
                    model, train_ds, val_ds, ckpt_path, seed)

            pred_df = predict_seq(model, test_df, available, TARGET, SEQ_LEN)
            pred_df["y_pred"] = pred_df["y_pred"].clip(lower=0)

            for lbl in ["normal", "abnormal"]:
                mask = pred_df["dataset_label"] == lbl
                if mask.sum() == 0:
                    continue
                mae  = mean_absolute_error(
                    pred_df.loc[mask,"y_true"], pred_df.loc[mask,"y_pred"])
                rmse = np.sqrt(mean_squared_error(
                    pred_df.loc[mask,"y_true"], pred_df.loc[mask,"y_pred"]))
                all_split_results.append({
                    "split"      : split_name,
                    "seed"       : seed,
                    "model"      : arch_name,
                    "feature_set": feat_name,
                    "flight_type": lbl,
                    "MAE_Wh"     : round(mae,  4),
                    "RMSE_Wh"    : round(rmse, 4),
                })
                print(f"    {arch_name} {feat_name} — {lbl.upper()} MAE: {mae:.4f} Wh")

            del model; gc.collect()
            if device.type == "mps":
                torch.mps.empty_cache()

        del train_ds, val_ds; gc.collect()

    del train_df, val_df, test_df, cnn_test; gc.collect()

results_df = pd.DataFrame(all_split_results)
results_df.to_csv(OUTPUT_DIR / "10split_raw_results.csv", index=False)
print(f"\n✓ Done. {len(results_df)} result rows saved.")



SPLIT: split_01_seed_11  SEED: 11
  CNN scores merged ✓  (134,655 rows)
  Train:(698993, 18) Val:(133277, 18) Test:(134655, 18)
  SKIP (checkpoint exists): LSTM_Base_13_split_01_seed_11
    LSTM Base_13 — NORMAL MAE: 0.2306 Wh
    LSTM Base_13 — ABNORMAL MAE: 1.0379 Wh
  SKIP (checkpoint exists): BiLSTM_Base_13_split_01_seed_11
    BiLSTM Base_13 — NORMAL MAE: 0.2665 Wh
    BiLSTM Base_13 — ABNORMAL MAE: 1.0672 Wh


KeyboardInterrupt: 

## 4. Summary Table — Mean ± Std MAE Across 10 Splits

In [ ]:
results_df = pd.read_csv(OUTPUT_DIR / "10split_raw_results.csv")

summary = (results_df
    .groupby(["model","feature_set","flight_type"])["MAE_Wh"]
    .agg(mean="mean", std="std", min="min", max="max", count="count")
    .round(4)
    .reset_index()
)

print("="*75)
print("10-SPLIT MEAN ± STD MAE — LSTM AND BiLSTM")
print("="*75)

for ftype in ["normal", "abnormal"]:
    sub = (summary[summary["flight_type"] == ftype]
           .sort_values("mean").reset_index(drop=True))
    sub.index += 1
    print(f"\n--- {ftype.upper()} FLIGHTS ---")
    print(sub[["model","feature_set","mean","std","min","max"]].to_string())

# Key finding
print("\n" + "="*75)
print("KEY FINDING: CNN score effect on ABNORMAL flights (mean across 10 splits)")
print("="*75)
abn = summary[summary["flight_type"] == "abnormal"]
for model in ["LSTM","BiLSTM"]:
    base = float(abn[(abn["model"]==model) &
                     (abn["feature_set"]=="Base_13")]["mean"])
    cnn  = float(abn[(abn["model"]==model) &
                     (abn["feature_set"]=="Base_13+CNN_Score")]["mean"])
    diff = base - cnn
    sign = "▼ IMPROVES" if diff > 0.01 else ("▲ WORSENS" if diff < -0.01 else "≈ NO CHANGE")
    print(f"  {model:<8} Base:{base:.4f}  +CNN:{cnn:.4f}  Δ={diff:+.4f}  {sign}")


## 5. Visualisation — Box Plots Across 10 Splits

In [ ]:
results_df = pd.read_csv(OUTPUT_DIR / "10split_raw_results.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Energy Prediction MAE Across 10 Random Splits\n"
             "LSTM and BiLSTM — Base vs CNN-Augmented Features",
             fontsize=13, fontweight="bold")

colors = {"Base_13": "#1976D2", "Base_13+CNN_Score": "#E53935"}
labels = {"Base_13": "Base 13 features", "Base_13+CNN_Score": "Base 13 + CNN score"}

for ax, ftype in zip(axes, ["normal", "abnormal"]):
    sub = results_df[results_df["flight_type"] == ftype]

    positions = []
    data      = []
    tick_labels = []
    pos = 1

    for model in ["LSTM", "BiLSTM"]:
        for feat in ["Base_13", "Base_13+CNN_Score"]:
            vals = sub[(sub["model"]==model) &
                       (sub["feature_set"]==feat)]["MAE_Wh"].values
            data.append(vals)
            positions.append(pos)
            tick_labels.append(f"{model}\n{labels[feat]}")
            pos += 1
        pos += 0.5  # gap between models

    bp = ax.boxplot(data, positions=positions, patch_artist=True,
                    widths=0.6, showfliers=True,
                    medianprops=dict(color="black", linewidth=2))

    feat_cycle = ["Base_13","Base_13+CNN_Score","Base_13","Base_13+CNN_Score"]
    for patch, feat in zip(bp["boxes"], feat_cycle):
        patch.set_facecolor(colors[feat])
        patch.set_alpha(0.75)

    ax.set_title(f"{ftype.upper()} flights", fontsize=11, fontweight="bold")
    ax.set_ylabel("MAE (Wh)", fontsize=10)
    ax.set_xticks(positions)
    ax.set_xticklabels(tick_labels, fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

    # Add mean labels
    for pos, vals in zip(positions, data):
        ax.text(pos, max(vals) + 0.02, f"{np.mean(vals):.3f}",
                ha="center", va="bottom", fontsize=8, color="#333")

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[f], alpha=0.75, label=labels[f])
                   for f in ["Base_13","Base_13+CNN_Score"]]
fig.legend(handles=legend_elements, loc="lower center",
           ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(PLOT_DIR / "10split_mae_boxplot.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: 10split_mae_boxplot.png")
plt.close()


In [ ]:
# ── Per-split line plot showing consistency across splits ────
results_df = pd.read_csv(OUTPUT_DIR / "10split_raw_results.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey="row")
fig.suptitle("MAE per Split — LSTM and BiLSTM\n"
             "Showing stability of CNN score effect across 10 splits",
             fontsize=12, fontweight="bold")

colors_feat = {"Base_13": "#1976D2", "Base_13+CNN_Score": "#E53935"}

for col, model in enumerate(["LSTM","BiLSTM"]):
    for row, ftype in enumerate(["normal","abnormal"]):
        ax = axes[row][col]
        sub = results_df[(results_df["model"]==model) &
                         (results_df["flight_type"]==ftype)]

        for feat in ["Base_13","Base_13+CNN_Score"]:
            s = sub[sub["feature_set"]==feat].sort_values("seed")
            ax.plot(range(1,11), s["MAE_Wh"].values,
                    marker="o", linewidth=1.8, markersize=6,
                    color=colors_feat[feat],
                    label=labels[feat])
            ax.axhline(s["MAE_Wh"].mean(),
                       color=colors_feat[feat], linewidth=1,
                       linestyle="--", alpha=0.6)

        ax.set_title(f"{model} — {ftype.upper()}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Split number", fontsize=9)
        ax.set_ylabel("MAE (Wh)", fontsize=9)
        ax.set_xticks(range(1,11))
        ax.grid(True, alpha=0.3)
        if row == 0 and col == 1:
            ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(PLOT_DIR / "10split_mae_per_split.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: 10split_mae_per_split.png")
plt.close()


## 6. Notes for Thesis

Results from this notebook replace the single-split results in notebook 06 as the
primary reported energy prediction performance.

Report as:
> "LSTM and BiLSTM were evaluated across 10 independent random splits.
>  Results are reported as mean ± std MAE (Wh) across splits."

Table format for thesis:
| Model | Feature set | Normal MAE | Abnormal MAE |
|-------|------------|-----------|-------------|
| LSTM  | Base 13    | X.XX ± X.XX | X.XX ± X.XX |
| LSTM  | +CNN score | X.XX ± X.XX | X.XX ± X.XX |
| BiLSTM | Base 13   | X.XX ± X.XX | X.XX ± X.XX |
| BiLSTM | +CNN score | X.XX ± X.XX | X.XX ± X.XX |
